# Advanced Repair Colab — Hard Negative Mining + Uncertainty

This notebook continues from your already trained `best_model.pth`.

It adds two advanced techniques:

1. **Hard Negative Mining**
   - Finds NORMAL images in the training split that the model tends to classify as PNEUMONIA.
   - Fine-tunes the last block of ResNet50 while giving these hard NORMAL images higher sampling weight.
   - Goal: reduce NORMAL false positives.

2. **Uncertainty / Abstention**
   - Instead of forcing every image into NORMAL or PNEUMONIA, the model can output `UNCERTAIN` for ambiguous images.
   - Goal: reduce dangerous/confident mistakes by marking unclear images for human review.

Important:
- This notebook does **not** use the test set for training.
- Hard negatives are mined from the training split only.
- Validation is used for threshold and uncertainty selection.
- Test is used only for final reporting.


## 0. Runtime

Recommended:

`Runtime → Change runtime type → GPU`

Then run all cells from top to bottom.


In [ ]:
!nvidia-smi || true


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Install dependencies

In [ ]:
!pip install -q kagglehub scikit-learn matplotlib pandas


## 3. Configuration

The notebook automatically searches for the most recent `best_model.pth` in Google Drive.

Edit the manual override cell if the wrong checkpoint is selected.


In [ ]:
from pathlib import Path
import os, time, json, math, zipfile
import numpy as np
import pandas as pd

MYDRIVE = Path('/content/drive/MyDrive')

priority = []
for folder_name in [
    'pneumonia_resnet50_FINAL_outputs',
    'pneumonia_resnet50_outputs',
    'outputs_resnet50_pneumonia',
]:
    p = MYDRIVE / folder_name / 'best_model.pth'
    if p.exists():
        priority.append(p)

candidates = priority if priority else list(MYDRIVE.rglob('best_model.pth'))
candidates = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)

print('Found best_model.pth candidates:')
for i, p in enumerate(candidates[:20]):
    print(f'{i}: {p} | modified: {time.ctime(p.stat().st_mtime)}')

if not candidates:
    raise FileNotFoundError('No best_model.pth found in Google Drive.')

CHECKPOINT_PATH = str(candidates[0])
BASE_OUTPUT_DIR = str(Path(CHECKPOINT_PATH).parent)
ADVANCED_OUTPUT_DIR = str(Path(BASE_OUTPUT_DIR) / 'advanced_repair_hnm_uncertainty')

Path(ADVANCED_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print('\nUsing:')
print('CHECKPOINT_PATH =', CHECKPOINT_PATH)
print('BASE_OUTPUT_DIR =', BASE_OUTPUT_DIR)
print('ADVANCED_OUTPUT_DIR =', ADVANCED_OUTPUT_DIR)


## 3B. Optional manual override

Only edit this if the automatic checkpoint is wrong.


In [ ]:
# Example:
# CHECKPOINT_PATH = '/content/drive/MyDrive/pneumonia_resnet50_outputs/best_model.pth'
# BASE_OUTPUT_DIR = '/content/drive/MyDrive/pneumonia_resnet50_outputs'
# ADVANCED_OUTPUT_DIR = '/content/drive/MyDrive/pneumonia_resnet50_outputs/advanced_repair_hnm_uncertainty'
# Path(ADVANCED_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print('CHECKPOINT_PATH =', CHECKPOINT_PATH)
print('BASE_OUTPUT_DIR =', BASE_OUTPUT_DIR)
print('ADVANCED_OUTPUT_DIR =', ADVANCED_OUTPUT_DIR)


## 4. Hyperparameters

These are conservative settings to avoid destroying the trained model.

Recommended first run:
- `HNM_EPOCHS = 3`
- `HARD_NEGATIVE_BOOST = 5.0`
- `NORMAL_LOSS_MULTIPLIER = 1.5`

If results improve but still need more, later try:
- `HNM_EPOCHS = 5`
- `HARD_NEGATIVE_BOOST = 8.0`


In [ ]:
# Hard Negative Mining
HNM_EPOCHS = 3
HNM_LR = 1e-5
BATCH_SIZE = 32
NUM_WORKERS = 0

HARD_NEGATIVE_BOOST = 5.0
NORMAL_LOSS_MULTIPLIER = 1.5
WEIGHT_DECAY = 1e-4

# Mining rule: NORMAL image is hard if prob(PNEUMONIA) is above this value.
# If too few are found, notebook automatically selects the highest-risk NORMAL images.
HARD_NEGATIVE_PROB_MIN = 0.50
MIN_HARD_NEGATIVES = 40

# Threshold selection after HNM
TARGET_SPECIFICITY = 0.90
MIN_SENSITIVITY = 0.95

# Uncertainty selection
UNCERTAINTY_MIN_COVERAGE = 0.80
UNCERTAINTY_MIN_SENSITIVITY_DECIDED = 0.95
UNCERTAINTY_MIN_SPECIFICITY_DECIDED = 0.90

print('Configured.')


## 5. Imports and helper functions

In [ ]:
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import datasets, models, transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    classification_report,
)

import matplotlib.pyplot as plt
from IPython.display import display, Image

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(42)

def find_chest_xray_root(start_path: Path) -> Path:
    start_path = Path(start_path)
    if (start_path / 'train').is_dir() and (start_path / 'test').is_dir():
        return start_path
    for root, dirs, files in os.walk(start_path):
        rp = Path(root)
        if 'train' in dirs and 'test' in dirs:
            if (rp/'train'/'NORMAL').is_dir() and (rp/'train'/'PNEUMONIA').is_dir():
                return rp
    raise FileNotFoundError(f'Could not find chest_xray root under {start_path}')

def get_dataset_root():
    common = [
        Path('/kaggle/input/chest-xray-pneumonia/chest_xray'),
        Path('/content/chest_xray'),
        Path('/content/drive/MyDrive/chest_xray'),
    ]
    for p in common:
        if p.exists():
            return find_chest_xray_root(p)
    import kagglehub
    downloaded = kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia')
    return find_chest_xray_root(Path(downloaded))

def build_model(model_name: str, num_classes: int):
    model_name = model_name.lower()
    if model_name == 'resnet18':
        model = models.resnet18(weights=None)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(0.0), nn.Linear(in_features, num_classes))
        return model
    if model_name == 'resnet50':
        model = models.resnet50(weights=None)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(0.0), nn.Linear(in_features, num_classes))
        return model
    if model_name == 'densenet121':
        model = models.densenet121(weights=None)
        in_features = model.classifier.in_features
        model.classifier = nn.Sequential(nn.Dropout(0.0), nn.Linear(in_features, num_classes))
        return model
    if model_name == 'efficientnet_b0':
        model = models.efficientnet_b0(weights=None)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(nn.Dropout(0.0), nn.Linear(in_features, num_classes))
        return model
    raise ValueError(f'Unsupported model: {model_name}')

def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False

def unfreeze_last_block_and_classifier(model, model_name):
    freeze_all(model)
    model_name = model_name.lower()
    if model_name.startswith('resnet'):
        for p in model.layer4.parameters():
            p.requires_grad = True
        for p in model.fc.parameters():
            p.requires_grad = True
    elif model_name == 'densenet121':
        for p in model.features.denseblock4.parameters():
            p.requires_grad = True
        for p in model.features.norm5.parameters():
            p.requires_grad = True
        for p in model.classifier.parameters():
            p.requires_grad = True
    elif model_name == 'efficientnet_b0':
        for block in model.features[-2:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in model.classifier.parameters():
            p.requires_grad = True
    else:
        raise ValueError(model_name)

def build_transforms(img_size=224):
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]

    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomApply([transforms.RandomRotation(degrees=7)], p=0.70),
        transforms.RandomApply([
            transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.95, 1.05))
        ], p=0.50),
        transforms.RandomHorizontalFlip(p=0.50),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])
    return train_tf, eval_tf

def make_datasets_and_splits(root, img_size=224, val_fraction=0.15, seed=42):
    train_tf, eval_tf = build_transforms(img_size)

    full_train_aug = datasets.ImageFolder(root/'train', transform=train_tf)
    full_train_eval = datasets.ImageFolder(root/'train', transform=eval_tf)
    test_eval = datasets.ImageFolder(root/'test', transform=eval_tf)

    targets = np.array(full_train_eval.targets)
    all_idx = np.arange(len(targets))

    train_idx, val_idx = train_test_split(
        all_idx,
        test_size=val_fraction,
        random_state=seed,
        stratify=targets,
    )

    return full_train_aug, full_train_eval, test_eval, train_idx, val_idx

def predict_probs(model, loader, device):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)[:, 1]
            ys.extend(labels.cpu().numpy().astype(int).tolist())
            ps.extend(probs.cpu().numpy().astype(float).tolist())
    return np.array(ys, dtype=int), np.array(ps, dtype=float)

def metric_row(y_true, y_prob, threshold, name=''):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = float('nan')

    return {
        'strategy': name,
        'threshold': float(threshold),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision_pneumonia': float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),
        'sensitivity_pneumonia': float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)),
        'specificity_normal': float(tn/(tn+fp) if (tn+fp) else 0.0),
        'f1_pneumonia': float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'roc_auc': float(auc),
        'tn': int(tn),
        'fp_normal_errors': int(fp),
        'fn_pneumonia_errors': int(fn),
        'tp': int(tp),
    }

def threshold_table(y_true, y_prob, thresholds):
    return pd.DataFrame([metric_row(y_true, y_prob, t) for t in thresholds])

def choose_threshold(df, target_specificity=0.90, min_sensitivity=0.95):
    d = df[(df['specificity_normal'] >= target_specificity) &
           (df['sensitivity_pneumonia'] >= min_sensitivity)].copy()
    if len(d):
        row = d.sort_values(['macro_f1', 'specificity_normal', 'sensitivity_pneumonia'], ascending=False).iloc[0]
        return float(row['threshold']), 'constraints_satisfied'

    d = df[df['specificity_normal'] >= target_specificity].copy()
    if len(d):
        row = d.sort_values(['sensitivity_pneumonia', 'macro_f1'], ascending=False).iloc[0]
        return float(row['threshold']), 'specificity_only_fallback'

    row = df.sort_values(['macro_f1', 'specificity_normal'], ascending=False).iloc[0]
    return float(row['threshold']), 'fallback_best_macro_f1'

def plot_confusion(cm, class_names, title, out_path):
    plt.figure()
    plt.imshow(cm, interpolation='nearest')
    plt.title(title)
    plt.colorbar()
    ticks = np.arange(len(class_names))
    plt.xticks(ticks, class_names, rotation=30, ha='right')
    plt.yticks(ticks, class_names)
    thresh = cm.max()/2 if cm.max() else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i,j]), ha='center', va='center',
                     color='white' if cm[i,j] > thresh else 'black')
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2), encoding='utf-8')

def atomic_save(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    torch.save(obj, tmp)
    tmp.replace(path)


## 6. Load model and datasets

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model_name = checkpoint.get('model_name', 'resnet50')
class_names = checkpoint.get('class_names', ['NORMAL', 'PNEUMONIA'])
img_size = int(checkpoint.get('img_size', 224))
checkpoint_threshold = float(checkpoint.get('threshold', 0.5))

print('model_name:', model_name)
print('class_names:', class_names)
print('img_size:', img_size)
print('checkpoint_threshold:', checkpoint_threshold)

root = get_dataset_root()
print('Dataset root:', root)

full_train_aug, full_train_eval, test_eval, train_idx, val_idx = make_datasets_and_splits(root, img_size=img_size)

train_eval_subset = Subset(full_train_eval, train_idx.tolist())
val_eval_subset = Subset(full_train_eval, val_idx.tolist())
test_eval_subset = test_eval

train_eval_loader = DataLoader(train_eval_subset, batch_size=64, shuffle=False, num_workers=0)
val_eval_loader = DataLoader(val_eval_subset, batch_size=64, shuffle=False, num_workers=0)
test_eval_loader = DataLoader(test_eval_subset, batch_size=64, shuffle=False, num_workers=0)

model = build_model(model_name, len(class_names))
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

print('Train split:', len(train_idx), 'Val split:', len(val_idx), 'Test:', len(test_eval))


## 7. Baseline evaluation and threshold repair before Hard Negative Mining

In [ ]:
thresholds = np.round(np.arange(0.01, 1.00, 0.01), 4)

val_y, val_p = predict_probs(model, val_eval_loader, device)
test_y, test_p = predict_probs(model, test_eval_loader, device)

val_table = threshold_table(val_y, val_p, thresholds)
test_table = threshold_table(test_y, test_p, thresholds)

baseline_threshold, baseline_reason = choose_threshold(
    val_table,
    target_specificity=TARGET_SPECIFICITY,
    min_sensitivity=MIN_SENSITIVITY,
)

baseline_old_metrics = metric_row(test_y, test_p, checkpoint_threshold, name='old_checkpoint_threshold')
baseline_repaired_metrics = metric_row(test_y, test_p, baseline_threshold, name='threshold_repaired_before_hnm')

print('Baseline old threshold metrics:')
print(json.dumps(baseline_old_metrics, indent=2))

print('\nBaseline repaired threshold from validation:')
print('threshold:', baseline_threshold, 'reason:', baseline_reason)
print(json.dumps(baseline_repaired_metrics, indent=2))

out_dir = Path(ADVANCED_OUTPUT_DIR)
val_table.to_csv(out_dir/'baseline_threshold_table_validation.csv', index=False)
test_table.to_csv(out_dir/'baseline_threshold_table_test.csv', index=False)
save_json(out_dir/'baseline_old_threshold_test_metrics.json', baseline_old_metrics)
save_json(out_dir/'baseline_repaired_threshold_test_metrics.json', baseline_repaired_metrics)


## 8. Mine hard NORMAL negatives from the training split

This uses the training split only.

A hard negative here means:

`true label = NORMAL`, but `prob(PNEUMONIA)` is high.


In [ ]:
train_y, train_p = predict_probs(model, train_eval_loader, device)
train_global_indices = np.array(train_idx)

normal_mask = (train_y == 0)
hard_mask = normal_mask & (train_p >= HARD_NEGATIVE_PROB_MIN)
hard_positions = np.where(hard_mask)[0]

if len(hard_positions) < MIN_HARD_NEGATIVES:
    # Take top-risk NORMAL images if strict condition gives too few.
    normal_positions = np.where(normal_mask)[0]
    order = normal_positions[np.argsort(train_p[normal_positions])[::-1]]
    hard_positions = order[:MIN_HARD_NEGATIVES]

hard_global_indices = set(train_global_indices[hard_positions].astype(int).tolist())

hard_df = pd.DataFrame({
    'subset_position': hard_positions.astype(int),
    'global_train_index': train_global_indices[hard_positions].astype(int),
    'true_label': [class_names[int(train_y[i])] for i in hard_positions],
    'prob_pneumonia': train_p[hard_positions],
})
hard_df = hard_df.sort_values('prob_pneumonia', ascending=False)
hard_df.to_csv(Path(ADVANCED_OUTPUT_DIR)/'mined_hard_normal_negatives_train_split.csv', index=False)

print('Hard NORMAL negatives mined:', len(hard_global_indices))
display(hard_df.head(30))


## 9. Build hard-negative weighted training loader

Hard NORMAL negatives get higher sampling probability.
NORMAL class also receives a slightly stronger loss multiplier.


In [ ]:
train_targets = np.array(full_train_eval.targets)[train_idx]
class_counts = np.bincount(train_targets, minlength=len(class_names))
class_weights = class_counts.sum() / (len(class_names) * np.maximum(class_counts, 1))
class_weights = class_weights.astype(np.float32)
class_weights[0] *= NORMAL_LOSS_MULTIPLIER

print('class_counts:', class_counts)
print('class_weights with NORMAL multiplier:', class_weights)

sample_weights = []
for global_idx in train_idx:
    y = full_train_eval.targets[int(global_idx)]
    w = float(class_weights[y])
    if int(global_idx) in hard_global_indices:
        w *= HARD_NEGATIVE_BOOST
    sample_weights.append(w)

sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

train_aug_subset = Subset(full_train_aug, train_idx.tolist())
train_hnm_loader = DataLoader(
    train_aug_subset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32).to(device))

print('Training batches:', len(train_hnm_loader))


## 10. Hard Negative Mining fine-tuning

This fine-tunes only the last block + classifier with a small learning rate.

It saves:
- `hnm_checkpoint_last.pth`
- `hnm_best_model.pth`
- `hnm_history.csv`


In [ ]:
def run_train_epoch(model, loader, criterion, optimizer, device, scaler=None):
    model.train()
    total_loss = 0.0
    total_items = 0
    ys, ps = [], []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        if scaler is not None and device.type == 'cuda':
            with torch.cuda.amp.autocast():
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 2.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 2.0)
            optimizer.step()

        probs = torch.softmax(logits.detach(), dim=1)[:, 1]
        bs = labels.size(0)
        total_loss += float(loss.item()) * bs
        total_items += bs
        ys.extend(labels.detach().cpu().numpy().astype(int).tolist())
        ps.extend(probs.detach().cpu().numpy().astype(float).tolist())

    return total_loss / max(total_items, 1), np.array(ys), np.array(ps)

# Start HNM from the original best model again.
hnm_model = build_model(model_name, len(class_names))
hnm_model.load_state_dict(checkpoint['model_state_dict'])
hnm_model.to(device)

unfreeze_last_block_and_classifier(hnm_model, model_name)

optimizer = optim.AdamW([p for p in hnm_model.parameters() if p.requires_grad], lr=HNM_LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

out_dir = Path(ADVANCED_OUTPUT_DIR)
resume_path = out_dir / 'hnm_checkpoint_last.pth'

start_epoch = 0
best_score = -1.0
best_threshold = baseline_threshold
history = []

if resume_path.exists():
    print('Resuming HNM from:', resume_path)
    r = torch.load(resume_path, map_location=device)
    hnm_model.load_state_dict(r['model_state_dict'])
    optimizer.load_state_dict(r['optimizer_state_dict'])
    if r.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(r['scheduler_state_dict'])
    if r.get('scaler_state_dict') is not None:
        scaler.load_state_dict(r['scaler_state_dict'])
    start_epoch = int(r.get('completed_epoch', 0))
    best_score = float(r.get('best_score', -1.0))
    best_threshold = float(r.get('best_threshold', baseline_threshold))
    history = list(r.get('history', []))
    print('completed_epoch:', start_epoch, 'best_score:', best_score, 'best_threshold:', best_threshold)

for epoch in range(start_epoch + 1, HNM_EPOCHS + 1):
    train_loss, tr_y, tr_p = run_train_epoch(hnm_model, train_hnm_loader, criterion, optimizer, device, scaler=scaler)

    # Evaluate validation
    val_y_h, val_p_h = predict_probs(hnm_model, val_eval_loader, device)
    val_table_h = threshold_table(val_y_h, val_p_h, thresholds)
    chosen_thr, reason = choose_threshold(val_table_h, TARGET_SPECIFICITY, MIN_SENSITIVITY)
    val_metrics = metric_row(val_y_h, val_p_h, chosen_thr, name='hnm_validation')

    score = val_metrics['macro_f1']
    scheduler.step(score)

    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'chosen_threshold': chosen_thr,
        'selection_reason': reason,
        **{f'val_{k}': v for k, v in val_metrics.items() if k not in ['strategy']},
    }
    history.append(row)

    print(f"Epoch {epoch}/{HNM_EPOCHS} | train_loss={train_loss:.4f} | val_macro_f1={score:.4f} | "
          f"val_sens={val_metrics['sensitivity_pneumonia']:.4f} | val_spec={val_metrics['specificity_normal']:.4f} | "
          f"thr={chosen_thr:.2f}")

    if score > best_score:
        best_score = float(score)
        best_threshold = float(chosen_thr)
        atomic_save({
            'model_state_dict': hnm_model.state_dict(),
            'model_name': model_name,
            'class_names': class_names,
            'img_size': img_size,
            'threshold': best_threshold,
            'val_metrics': val_metrics,
            'hard_negative_boost': HARD_NEGATIVE_BOOST,
            'normal_loss_multiplier': NORMAL_LOSS_MULTIPLIER,
            'hnm_epoch': epoch,
        }, out_dir/'hnm_best_model.pth')
        print('  Saved new hnm_best_model.pth')

    atomic_save({
        'model_state_dict': hnm_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'completed_epoch': epoch,
        'best_score': best_score,
        'best_threshold': best_threshold,
        'history': history,
    }, out_dir/'hnm_checkpoint_last.pth')

    pd.DataFrame(history).to_csv(out_dir/'hnm_history.csv', index=False)

print('HNM finished. best_threshold:', best_threshold, 'best_score:', best_score)


## 11. Evaluate HNM model on test

In [ ]:
out_dir = Path(ADVANCED_OUTPUT_DIR)
hnm_best_path = out_dir / 'hnm_best_model.pth'

if not hnm_best_path.exists():
    raise FileNotFoundError('hnm_best_model.pth was not saved.')

hnm_ckpt = torch.load(hnm_best_path, map_location=device)
final_model = build_model(hnm_ckpt['model_name'], len(hnm_ckpt['class_names']))
final_model.load_state_dict(hnm_ckpt['model_state_dict'])
final_model.to(device)
final_model.eval()

hnm_threshold = float(hnm_ckpt.get('threshold', best_threshold))
print('HNM selected threshold:', hnm_threshold)

val_y_final, val_p_final = predict_probs(final_model, val_eval_loader, device)
test_y_final, test_p_final = predict_probs(final_model, test_eval_loader, device)

hnm_val_table = threshold_table(val_y_final, val_p_final, thresholds)
hnm_test_table = threshold_table(test_y_final, test_p_final, thresholds)
hnm_val_table.to_csv(out_dir/'hnm_threshold_table_validation.csv', index=False)
hnm_test_table.to_csv(out_dir/'hnm_threshold_table_test.csv', index=False)

hnm_test_metrics = metric_row(test_y_final, test_p_final, hnm_threshold, name='hard_negative_mining')
baseline_test_metrics = metric_row(test_y, test_p, baseline_threshold, name='baseline_threshold_repair')

comparison_df = pd.DataFrame([baseline_test_metrics, hnm_test_metrics])
comparison_df.to_csv(out_dir/'baseline_vs_hnm_test_comparison.csv', index=False)
display(comparison_df)

save_json(out_dir/'hnm_test_metrics.json', hnm_test_metrics)

test_pred_hnm = (test_p_final >= hnm_threshold).astype(int)
cm_hnm = confusion_matrix(test_y_final, test_pred_hnm, labels=[0,1])
plot_confusion(cm_hnm, class_names, f'HNM test confusion matrix, threshold={hnm_threshold:.2f}', out_dir/'confusion_matrix_test_hnm.png')

report = classification_report(test_y_final, test_pred_hnm, target_names=class_names, zero_division=0)
print(report)
Path(out_dir/'classification_report_test_hnm.txt').write_text(report, encoding='utf-8')

display(Image(filename=str(out_dir/'confusion_matrix_test_hnm.png')))


## 12. Uncertainty / Abstention selection

This creates a three-output decision:

- `NORMAL` if `prob(PNEUMONIA) <= low_threshold`
- `PNEUMONIA` if `prob(PNEUMONIA) >= high_threshold`
- `UNCERTAIN` if the probability is between both thresholds

This is useful for ambiguous X-ray images.


In [ ]:
def uncertainty_row(y_true, y_prob, low, high, name=''):
    # -1 = uncertain, 0 = NORMAL, 1 = PNEUMONIA
    pred = np.full_like(y_true, fill_value=-1)
    pred[y_prob <= low] = 0
    pred[y_prob >= high] = 1

    decided = pred != -1
    uncertain = ~decided

    coverage = decided.mean()
    uncertain_rate = uncertain.mean()
    uncertain_count = int(uncertain.sum())

    if decided.sum() == 0:
        return {
            'strategy': name, 'low_threshold': low, 'high_threshold': high,
            'coverage': 0.0, 'uncertain_rate': 1.0, 'uncertain_count': len(y_true),
            'accuracy_decided': 0.0, 'sensitivity_pneumonia_decided': 0.0, 'specificity_normal_decided': 0.0,
            'macro_f1_decided': 0.0,
            'normal_fp_decided': 0, 'pneumonia_fn_decided': 0,
            'normal_uncertain': int(((y_true == 0) & uncertain).sum()),
            'pneumonia_uncertain': int(((y_true == 1) & uncertain).sum()),
        }

    yd = y_true[decided]
    pd = pred[decided]

    cm = confusion_matrix(yd, pd, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()

    return {
        'strategy': name,
        'low_threshold': float(low),
        'high_threshold': float(high),
        'coverage': float(coverage),
        'uncertain_rate': float(uncertain_rate),
        'uncertain_count': uncertain_count,
        'accuracy_decided': float(accuracy_score(yd, pd)),
        'sensitivity_pneumonia_decided': float(recall_score(yd, pd, pos_label=1, zero_division=0)),
        'specificity_normal_decided': float(tn/(tn+fp) if (tn+fp) else 0.0),
        'macro_f1_decided': float(f1_score(yd, pd, average='macro', zero_division=0)),
        'normal_fp_decided': int(fp),
        'pneumonia_fn_decided': int(fn),
        'normal_uncertain': int(((y_true == 0) & uncertain).sum()),
        'pneumonia_uncertain': int(((y_true == 1) & uncertain).sum()),
    }

def build_uncertainty_table(y_true, y_prob):
    lows = np.round(np.arange(0.01, 0.51, 0.01), 4)
    highs = np.round(np.arange(0.50, 1.00, 0.01), 4)
    rows = []
    for low in lows:
        for high in highs:
            if low < high:
                rows.append(uncertainty_row(y_true, y_prob, low, high))
    return pd.DataFrame(rows)

unc_val_table = build_uncertainty_table(val_y_final, val_p_final)

valid_unc = unc_val_table[
    (unc_val_table['coverage'] >= UNCERTAINTY_MIN_COVERAGE) &
    (unc_val_table['sensitivity_pneumonia_decided'] >= UNCERTAINTY_MIN_SENSITIVITY_DECIDED) &
    (unc_val_table['specificity_normal_decided'] >= UNCERTAINTY_MIN_SPECIFICITY_DECIDED)
].copy()

if len(valid_unc):
    selected_unc = valid_unc.sort_values(['coverage', 'macro_f1_decided', 'specificity_normal_decided'], ascending=False).iloc[0]
    reason_unc = 'constraints_satisfied'
else:
    selected_unc = unc_val_table.sort_values(['macro_f1_decided', 'coverage'], ascending=False).iloc[0]
    reason_unc = 'fallback_best_macro_f1_decided'

low_thr = float(selected_unc['low_threshold'])
high_thr = float(selected_unc['high_threshold'])

unc_test_metrics = uncertainty_row(test_y_final, test_p_final, low_thr, high_thr, name='hnm_uncertainty_abstention')

unc_val_table.to_csv(out_dir/'uncertainty_table_validation.csv', index=False)
save_json(out_dir/'uncertainty_selected_validation_rule.json', {
    'selection_reason': reason_unc,
    'low_threshold': low_thr,
    'high_threshold': high_thr,
    'selected_validation_row': selected_unc.to_dict(),
    'test_metrics': unc_test_metrics,
})

print('Selected uncertainty rule:')
print('reason:', reason_unc)
print('low_threshold:', low_thr, 'high_threshold:', high_thr)
print('\\nTest uncertainty metrics:')
print(json.dumps(unc_test_metrics, indent=2))


## 13. Save uncertainty predictions and show uncertainty summary

In [ ]:
def save_uncertainty_predictions(path, y_true, y_prob, low, high, class_names):
    pred = np.full_like(y_true, fill_value=-1)
    pred[y_prob <= low] = 0
    pred[y_prob >= high] = 1

    def label_name(v):
        if v == -1:
            return 'UNCERTAIN'
        return class_names[int(v)]

    df = pd.DataFrame({
        'true_index': y_true,
        'true_label': [class_names[int(i)] for i in y_true],
        'prob_pneumonia': y_prob,
        'decision_index': pred,
        'decision_label': [label_name(v) for v in pred],
        'low_threshold': low,
        'high_threshold': high,
        'is_uncertain': pred == -1,
        'is_decided_error': (pred != -1) & (pred != y_true),
    })
    df.to_csv(path, index=False)
    return df

unc_df = save_uncertainty_predictions(
    out_dir/'test_predictions_hnm_uncertainty.csv',
    test_y_final,
    test_p_final,
    low_thr,
    high_thr,
    class_names,
)

print('Decision counts:')
display(unc_df['decision_label'].value_counts().to_frame())

print('Uncertain examples:')
display(unc_df[unc_df['is_uncertain']].sort_values('prob_pneumonia').head(20))

print('Decided errors:')
display(unc_df[unc_df['is_decided_error']].sort_values('prob_pneumonia').head(30))


## 14. Final comparison table

In [ ]:
final_rows = [
    baseline_test_metrics,
    hnm_test_metrics,
]

final_binary_df = pd.DataFrame(final_rows)
final_binary_df.to_csv(out_dir/'final_binary_comparison_baseline_vs_hnm.csv', index=False)

unc_summary_df = pd.DataFrame([unc_test_metrics])
unc_summary_df.to_csv(out_dir/'final_uncertainty_summary.csv', index=False)

print('Binary classification comparison:')
display(final_binary_df)

print('Uncertainty / abstention summary:')
display(unc_summary_df)

# Plot threshold curves after HNM
plt.figure()
plt.plot(hnm_val_table['threshold'], hnm_val_table['sensitivity_pneumonia'], label='val sensitivity pneumonia')
plt.plot(hnm_val_table['threshold'], hnm_val_table['specificity_normal'], label='val specificity normal')
plt.plot(hnm_val_table['threshold'], hnm_val_table['macro_f1'], label='val macro F1')
plt.axvline(hnm_threshold, linestyle='--', label=f'HNM threshold {hnm_threshold:.2f}')
plt.xlabel('Threshold')
plt.ylabel('Metric')
plt.title('HNM validation metrics by threshold')
plt.legend()
plt.tight_layout()
plt.savefig(out_dir/'hnm_threshold_curves_validation.png', dpi=200)
plt.close()

plt.figure()
plt.plot(hnm_test_table['threshold'], hnm_test_table['sensitivity_pneumonia'], label='test sensitivity pneumonia')
plt.plot(hnm_test_table['threshold'], hnm_test_table['specificity_normal'], label='test specificity normal')
plt.plot(hnm_test_table['threshold'], hnm_test_table['macro_f1'], label='test macro F1')
plt.axvline(hnm_threshold, linestyle='--', label=f'HNM threshold {hnm_threshold:.2f}')
plt.xlabel('Threshold')
plt.ylabel('Metric')
plt.title('HNM test metrics by threshold')
plt.legend()
plt.tight_layout()
plt.savefig(out_dir/'hnm_threshold_curves_test.png', dpi=200)
plt.close()

for name in [
    'confusion_matrix_test_hnm.png',
    'hnm_threshold_curves_validation.png',
    'hnm_threshold_curves_test.png',
]:
    print('\n' + name)
    display(Image(filename=str(out_dir/name)))


## 15. Zip all advanced results

Download the ZIP and send it back.


In [ ]:
out_dir = Path(ADVANCED_OUTPUT_DIR)
zip_path = Path('/content/advanced_repair_hnm_uncertainty_results.zip')

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in out_dir.glob('*'):
        if p.is_file():
            z.write(p, arcname=p.name)

print('Created:', zip_path)

from google.colab import files
files.download(str(zip_path))
